# 01.6 - Heaps & Priority Queues

## What is a heap?
A **complete binary tree** where every parent is **≤ its children** (min-heap) or **≥** (max-heap).
So the **smallest (or largest) element is always at the root** → O(1) peek.

It's stored in a plain **array** - no pointers needed:
```
            1                index:  0  1  2  3  4  5
          /   \              array: [1, 3, 2, 7, 4, 5]
         3     2
        / \   /              parent(i) = (i - 1) // 2
       7   4 5               children(i) = 2i + 1, 2i + 2
```

A **priority queue** is the abstract idea ("always give me the most important item"); a heap is how it's implemented.

## Where is it used?
| Real system | Use |
|---|---|
| **Operating systems** | CPU scheduling - run the highest-priority process |
| **Google Maps / network routing** | **Dijkstra's shortest path** (see 02.4) |
| **Hospital ER, ride-hailing dispatch** | Most urgent / nearest first |
| **Trending / Top-K** (Twitter trends, top products, leaderboards) | Keep a heap of size k |
| **Event simulation, timers, job schedulers (cron, Celery)** | Next event = smallest timestamp |
| **Median of a data stream** (monitoring latency) | Two heaps |
| **Compression** | Huffman coding (see 02.5) |
| **Merging sorted files** (databases, MapReduce, external sort) | Merge k sorted lists |

## Complexity
| Operation | Time |
|---|---|
| peek min | O(1) |
| push / pop | O(log n) |
| build heap from list (`heapify`) | O(n) |
| heap sort | O(n log n) |

## 1. Build a min-heap from scratch

In [1]:
class MinHeap:
    def __init__(self):
        self.a = []

    def push(self, val):                 # O(log n): add at end, bubble up
        self.a.append(val)
        i = len(self.a) - 1
        while i > 0:
            parent = (i - 1) // 2
            if self.a[i] < self.a[parent]:
                self.a[i], self.a[parent] = self.a[parent], self.a[i]
                i = parent
            else:
                break

    def pop(self):                       # O(log n): move last to root, sift down
        a = self.a
        if not a:
            raise IndexError("pop from empty heap")
        top = a[0]
        last = a.pop()
        if a:
            a[0] = last
            i = 0
            while True:
                smallest = i
                for child in (2 * i + 1, 2 * i + 2):
                    if child < len(a) and a[child] < a[smallest]:
                        smallest = child
                if smallest == i:
                    break
                a[i], a[smallest] = a[smallest], a[i]
                i = smallest
        return top

    def peek(self):
        return self.a[0]

    def __len__(self):
        return len(self.a)

h = MinHeap()
for x in [5, 3, 8, 1, 9, 2]:
    h.push(x)
print("internal array:", h.a)
print("pop order:", [h.pop() for _ in range(len(h))])   # sorted! (this is heap sort)

internal array: [1, 3, 2, 5, 9, 8]
pop order: [1, 2, 3, 5, 8, 9]


## 2. Python's `heapq` (min-heap on a list)

In [2]:
import heapq

nums = [5, 3, 8, 1, 9, 2]
heapq.heapify(nums)                    # O(n)
heapq.heappush(nums, 0)
print(heapq.heappop(nums), nums[0])    # 0 then the new min 1
print(heapq.nlargest(3, [5, 3, 8, 1, 9, 2]), heapq.nsmallest(2, [5, 3, 8, 1]))

# Max-heap: push negatives
mx = []
for x in [5, 3, 8]:
    heapq.heappush(mx, -x)
print("max:", -mx[0])

# Priority queue of tasks: (priority, counter, task) - counter breaks ties safely
import itertools
counter = itertools.count()
pq = []
for pri, task in [(3, "email"), (1, "server down!"), (2, "code review")]:
    heapq.heappush(pq, (pri, next(counter), task))
while pq:
    pri, _, task = heapq.heappop(pq)
    print(pri, task)

0 1
[9, 8, 5] [1, 3]
max: 8
1 server down!
2 code review
3 email


---
## Solved problems

### 1. Last Stone Weight - LC #1046 (Easy)
Smash the two heaviest stones; if different, the difference remains. Return the last stone (or 0).

In [3]:
import heapq

def last_stone_weight(stones):
    heap = [-s for s in stones]          # max-heap
    heapq.heapify(heap)
    while len(heap) > 1:
        a = -heapq.heappop(heap)
        b = -heapq.heappop(heap)
        if a != b:
            heapq.heappush(heap, -(a - b))
    return -heap[0] if heap else 0

assert last_stone_weight([2, 7, 4, 1, 8, 1]) == 1
assert last_stone_weight([1]) == 1
print("Last Stone Weight ")

Last Stone Weight 


### 2. Kth Largest Element in a Stream - LC #703 (Easy)
Class that returns the k-th largest after each `add`. (Live leaderboards.)

**Idea:** Min-heap of size k: its root is the k-th largest.

In [4]:
import heapq

class KthLargest:
    def __init__(self, k, nums):
        self.k = k
        self.heap = []
        for x in nums:
            self.add(x)

    def add(self, val):
        heapq.heappush(self.heap, val)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)       # keep only the k largest
        return self.heap[0]

kl = KthLargest(3, [4, 5, 8, 2])
assert [kl.add(x) for x in [3, 5, 10, 9, 4]] == [4, 5, 5, 8, 8]
print("Kth Largest in Stream ")

Kth Largest in Stream 


### 3. Merge k Sorted Lists - LC #23 (Hard) Merge k sorted linked lists into one. (How databases merge sorted runs.)

**Idea:** Heap holds the current head of each list; pop the smallest, push its next.

In [5]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(values):
    """[1,2,3] -> 1 -> 2 -> 3  (test helper, not needed on LeetCode)"""
    dummy = cur = ListNode()
    for v in values:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def list_to_py(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

import heapq

def merge_k_lists(lists):
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))   # i breaks ties (nodes aren't comparable)
    dummy = tail = ListNode()
    while heap:
        val, i, node = heapq.heappop(heap)
        tail.next = node
        tail = node
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

# Time O(N log k), N = total nodes
lists = [build_list([1, 4, 5]), build_list([1, 3, 4]), build_list([2, 6])]
assert list_to_py(merge_k_lists(lists)) == [1, 1, 2, 3, 4, 4, 5, 6]
assert merge_k_lists([]) is None
print("Merge k Lists ")

Merge k Lists 


### 4. Find Median from Data Stream - LC #295 (Hard) Add numbers one by one; get the median at any time in O(1). (Latency monitoring: p50.)

**Idea:** Two heaps: a max-heap for the smaller half and a min-heap for the larger half. The median is at their tops.

In [6]:
import heapq

class MedianFinder:
    def __init__(self):
        self.low = []     # max-heap (negated): smaller half
        self.high = []    # min-heap: larger half

    def addNum(self, num):
        heapq.heappush(self.low, -num)
        heapq.heappush(self.high, -heapq.heappop(self.low))   # move largest of low to high
        if len(self.high) > len(self.low):                     # keep low >= high in size
            heapq.heappush(self.low, -heapq.heappop(self.high))

    def findMedian(self):
        if len(self.low) > len(self.high):
            return -self.low[0]
        return (-self.low[0] + self.high[0]) / 2

mf = MedianFinder()
mf.addNum(1); mf.addNum(2)
assert mf.findMedian() == 1.5
mf.addNum(3)
assert mf.findMedian() == 2
for x in [10, 11]:
    mf.addNum(x)
assert mf.findMedian() == 3
print("Median Finder ")

Median Finder 


### 5. Task Scheduler - LC #621 (Medium)
CPU tasks with a cooldown `n` between identical tasks. Minimum time units (idle allowed) to finish all.

**Idea:** Greedy simulation: always run the task with the most remaining copies (max-heap); tasks waiting for cooldown sit in a queue.

In [7]:
import heapq
from collections import Counter, deque

def least_interval(tasks, n):
    heap = [-c for c in Counter(tasks).values()]    # max-heap of remaining counts
    heapq.heapify(heap)
    cooldown = deque()                               # (count, time_available)
    time = 0
    while heap or cooldown:
        time += 1
        if heap:
            remaining = heapq.heappop(heap) + 1      # run the most frequent task
            if remaining:
                cooldown.append((remaining, time + n))
        if cooldown and cooldown[0][1] == time:
            heapq.heappush(heap, cooldown.popleft()[0])
    return time

assert least_interval(["A", "A", "A", "B", "B", "B"], 2) == 8
assert least_interval(["A", "C", "A", "B", "D", "B"], 1) == 6
assert least_interval(["A", "A", "A", "B", "B", "B"], 0) == 6
print("Task Scheduler ")

Task Scheduler 


## More practice on LeetCode
Easy: Relative Ranks (#506)
Medium: Top K Frequent Words (#692), Reorganize String (#767), Design Twitter (#355), Single-Threaded CPU (#1834)
Hard: IPO (#502), Sliding Window Median (#480)

Also in `03-Interview Patterns/03.1`: Kth Largest Element, K Closest Points, Top K Frequent.

## Key takeaways
- Heap = "give me the min/max quickly, while items keep changing".
- `heapq` is a **min-heap**; negate for max-heap; use tuples `(priority, tiebreak, item)`.
- **Top-k** → heap of size k. **Median** → two heaps. **Merge k sorted** → heap of heads.

**Next: `01.7-Graphs.ipynb`**